# Two-Tower Broker Match Model — Fixed

Key changes from original:
- **Focal Loss** replaces BCE + IPS (single robust solution for imbalance)
- **WeightedRandomSampler** — balanced batches so model sees enough positives
- **Expanded fusion head** — uses full embeddings, not just dot product scalar
- **propensity_score removed** from INTERACTION_FEATURES (was leaking into features + weights)
- **Threshold tuning** on val set instead of hardcoded 0.5
- **Gradient check** after epoch 1 to catch silent failures
- **Overfit sanity test** before full training

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, f1_score


In [3]:
df = pd.read_csv("prepared_positive_v81.csv")
print(f"Loaded: {df.shape}")
print(f"Conversion rate: {df['converted'].mean()*100:.2f}%")

CLIENT_FEATURES = [
    "quote_value_scaled",
    "lead_difficulty_scaled",
    "sophistication_scaled",
    "patience_hours_scaled",
    "digital_engagement_score_scaled",
    "tenure_years_scaled",
    "log_quote_value",
    "log_patience_hours",
    "month",
    "hour_of_day",
    "lead_dayofweek",
    "lead_quarter",
    "is_weekend",
    "insurance_type_enc",
    "claims_risk",
    "multi_product_intent",
    "insurance_type_missing",
    "language_missing",
    "tenure_years_missing",
    "digital_engagement_score_missing",
]

BROKER_FEATURES = [
    "skill_level_scaled",
    "conversion_rate_scaled",
    "csat_score_scaled",
    "reliability_scaled",
    "efficiency_scaled",
    "avg_response_time_scaled",
    "burnout_risk_scaled",
    "commission_rate_scaled",
    "cost_per_lead_scaled",
    "utilization_scaled",
    "ribo_experience",
    "ribo_licensed",
    "is_new_broker",
    "expertise_auto",
    "expertise_home",
    "expertise_bundle",
    "broker_quality_score",
]

# ── propensity_score REMOVED — it was leaking (used both as feature AND IPS weight)
INTERACTION_FEATURES = [
    "expertise_match",
    "language_match",
    "workload_ratio",
    "quality_x_value",
    "position_bias",
    "interaction_number",
]


Loaded: (20354, 116)
Conversion rate: 11.68%


/tmp/ipykernel_61476/3111640493.py:1: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("prepared_positive_v81.csv")


In [4]:
SEED           = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

EMBEDDING_DIM  = 64
HIDDEN_DIM     = 128
DROPOUT        = 0.3
LEARNING_RATE  = 1e-3
BATCH_SIZE     = 512
EPOCHS         = 30
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Focal loss params — tune alpha if precision/recall balance is off
FOCAL_ALPHA    = 0.75   # weight for positive class; increase if recall too low
FOCAL_GAMMA    = 2.0    # focus on hard examples; 2.0 is a solid default

print(f"Device: {DEVICE}")


Device: cuda


In [5]:
def filter_cols(cols, df):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"  Warning — missing columns (skipped): {missing}")
    return [c for c in cols if c in df.columns]

CLIENT_FEATURES      = filter_cols(CLIENT_FEATURES, df)
BROKER_FEATURES      = filter_cols(BROKER_FEATURES, df)
INTERACTION_FEATURES = filter_cols(INTERACTION_FEATURES, df)

print(f"Client features:      {len(CLIENT_FEATURES)}")
print(f"Broker features:      {len(BROKER_FEATURES)}")
print(f"Interaction features: {len(INTERACTION_FEATURES)}")

all_features = CLIENT_FEATURES + BROKER_FEATURES + INTERACTION_FEATURES + ["converted", "propensity_score"]
all_features = [c for c in all_features if c in df.columns]  # safe guard
df = df[all_features].fillna(0)


  Warning — missing columns (skipped): ['ribo_experience']
Client features:      20
Broker features:      16
Interaction features: 6


In [6]:
df_full = pd.read_csv("prepared_positive_v81.csv")
df_full = df_full.fillna(0)

if "lead_year" in df_full.columns and "lead_quarter" in df_full.columns:
    df_full["_time_key"] = df_full["lead_year"] * 10 + df_full["lead_quarter"]
    df_full = df_full.sort_values("_time_key").reset_index(drop=True)
    n = len(df_full)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.85)
    train_df  = df_full.iloc[:train_end].copy()
    val_df    = df_full.iloc[train_end:val_end].copy()
    test_df   = df_full.iloc[val_end:].copy()
    print("Temporal split:")
else:
    print("Warning: no temporal features found — using random split")
    train_df, temp_df = train_test_split(df_full, test_size=0.30, random_state=SEED,
                                          stratify=df_full["converted"])
    val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=SEED,
                                          stratify=temp_df["converted"])
    print("Random split:")

print(f"  Train: {len(train_df):,}  ({train_df['converted'].mean()*100:.1f}% positive)")
print(f"  Val:   {len(val_df):,}  ({val_df['converted'].mean()*100:.1f}% positive)")
print(f"  Test:  {len(test_df):,}  ({test_df['converted'].mean()*100:.1f}% positive)")


Temporal split:
  Train: 14,247  (11.9% positive)
  Val:   3,053  (11.2% positive)
  Test:  3,054  (11.2% positive)


/tmp/ipykernel_61476/1989140280.py:1: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv("prepared_positive_v81.csv")


In [7]:
def smart_convert_to_numeric(series, column_name):
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(float)
    converted = pd.to_numeric(series, errors='coerce')
    if converted.isna().sum() > len(series) * 0.1:
        unique_vals = series.dropna().unique()
        if len(unique_vals) <= 5:
            mapping = {val: idx for idx, val in enumerate(sorted(unique_vals))}
            converted = series.map(mapping)
        else:
            converted = pd.Series(0, index=series.index)
    return converted.fillna(0).astype(float)


class BrokerMatchDataset(Dataset):
    def __init__(self, df):
        self.client_x      = self._create_tensor(df, CLIENT_FEATURES,      "client")
        self.broker_x      = self._create_tensor(df, BROKER_FEATURES,      "broker")
        self.interaction_x = self._create_tensor(df, INTERACTION_FEATURES, "interaction")
        self.labels        = self._create_tensor(df, ["converted"],        "labels").squeeze()

    def _create_tensor(self, df, columns, name):
        existing_cols = [c for c in columns if c in df.columns]
        if not existing_cols:
            return torch.zeros(len(df), 1, dtype=torch.float32)
        df_subset = pd.DataFrame()
        for col in existing_cols:
            df_subset[col] = smart_convert_to_numeric(df[col], col)
        return torch.tensor(df_subset.values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.client_x[idx],
            self.broker_x[idx],
            self.interaction_x[idx],
            self.labels[idx],
        )


train_ds = BrokerMatchDataset(train_df)
val_ds   = BrokerMatchDataset(val_df)
test_ds  = BrokerMatchDataset(test_df)

# ── WeightedRandomSampler: balanced batches (fixes model collapse) ──────────
labels_array   = train_df["converted"].values.astype(int)
class_counts   = np.bincount(labels_array)          # [n_neg, n_pos]
class_weights  = 1.0 / class_counts                 # inverse frequency
sample_weights = class_weights[labels_array]        # per-sample weight

sampler = WeightedRandomSampler(
    weights     = torch.tensor(sample_weights, dtype=torch.float32),
    num_samples = len(sample_weights),
    replacement = True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,    num_workers=0)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")
print(f"Class counts — neg: {class_counts[0]:,}  pos: {class_counts[1]:,}  ratio: {class_counts[0]/class_counts[1]:.1f}x")


Train batches: 28
Val batches:   6
Test batches:  6
Class counts — neg: 12,552  pos: 1,695  ratio: 7.4x


In [8]:
class Tower(nn.Module):
    def __init__(self, input_dim, hidden_dim=HIDDEN_DIM, embed_dim=EMBEDDING_DIM, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, embed_dim),
        )

    def forward(self, x):
        emb = self.net(x)
        return nn.functional.normalize(emb, p=2, dim=1)


class TwoTowerModel(nn.Module):
    """
    Fix: fusion head now receives full embeddings (not just dot scalar).
    Input to head = [client_emb (64) | broker_emb (64) | hadamard (64) | dot (1) | interaction (N)]
    """
    def __init__(self, client_dim, broker_dim, interaction_dim, embed_dim=EMBEDDING_DIM):
        super().__init__()
        self.client_tower = Tower(client_dim, embed_dim=embed_dim)
        self.broker_tower = Tower(broker_dim, embed_dim=embed_dim)

        # embed + embed + hadamard + dot + interactions
        fusion_input_dim = embed_dim + embed_dim + embed_dim + 1 + interaction_dim
        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_input_dim, 64),
            nn.ReLU(),
            nn.Dropout(DROPOUT * 0.5),
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, client_x, broker_x, interaction_x):
        client_emb = self.client_tower(client_x)            # (B, 64)
        broker_emb = self.broker_tower(broker_x)            # (B, 64)
        hadamard   = client_emb * broker_emb               # (B, 64) element-wise product
        dot        = hadamard.sum(dim=1, keepdim=True)     # (B, 1)  cosine similarity
        fusion_input = torch.cat([client_emb, broker_emb, hadamard, dot, interaction_x], dim=1)
        return self.fusion_head(fusion_input).squeeze(1)   # raw logit

    def get_embeddings(self, client_x, broker_x):
        self.eval()
        with torch.no_grad():
            return self.client_tower(client_x), self.broker_tower(broker_x)


model = TwoTowerModel(
    client_dim      = len(CLIENT_FEATURES),
    broker_dim      = len(BROKER_FEATURES),
    interaction_dim = len(INTERACTION_FEATURES),
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")


TwoTowerModel(
  (client_tower): Tower(
    (net): Sequential(
      (0): Linear(in_features=20, out_features=128, bias=True)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): Dropout(p=0.15, inplace=False)
      (8): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (broker_tower): Tower(
    (net): Sequential(
      (0): Linear(in_features=16, out_features=128, bias=True)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
   

In [9]:
class FocalLoss(nn.Module):
    """
    Focal Loss — replaces BCE + pos_weight + IPS with one robust solution.
    - alpha: weight for positive class (increase toward 1.0 if recall too low)
    - gamma: focus on hard examples (0=BCE, 2=standard focal, 5=very aggressive)
    """
    def __init__(self, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, labels):
        bce    = F.binary_cross_entropy_with_logits(logits, labels, reduction='none')
        pt     = torch.exp(-bce)                                       # p(correct class)
        alpha_t = self.alpha * labels + (1 - self.alpha) * (1 - labels)
        focal  = alpha_t * (1 - pt) ** self.gamma * bce
        return focal.mean()


loss_fn   = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5   # patience=5 (was 3, too aggressive)
)

print(f"Focal Loss: alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA}")
print(f"Optimizer: AdamW  lr={LEARNING_RATE}  weight_decay=1e-4")


Focal Loss: alpha=0.75, gamma=2.0
Optimizer: AdamW  lr=0.001  weight_decay=1e-4


## Overfit Sanity Test
Before full training, verify the model *can* learn. Should reach AUC > 0.85 on 100 balanced samples within 50 epochs.
If it can't — the model has a fundamental bug (gradient flow, loss sign, label dtype, etc.).

In [10]:
tiny_pos  = train_df[train_df["converted"] == 1].head(50)
tiny_neg  = train_df[train_df["converted"] == 0].head(50)
tiny_df   = pd.concat([tiny_pos, tiny_neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)

tiny_ds     = BrokerMatchDataset(tiny_df)
tiny_loader = DataLoader(tiny_ds, batch_size=16, shuffle=True)

# Fresh model + loss for the test (don't pollute main model)
_test_model = TwoTowerModel(
    client_dim      = len(CLIENT_FEATURES),
    broker_dim      = len(BROKER_FEATURES),
    interaction_dim = len(INTERACTION_FEATURES),
).to(DEVICE)
_test_loss_fn  = FocalLoss()
_test_optimizer = torch.optim.AdamW(_test_model.parameters(), lr=1e-3)

print("Overfit test (50 pos + 50 neg samples):")
print(f"{'Epoch':>6}  {'Loss':>8}  {'AUC':>8}")

for ep in range(1, 51):
    _test_model.train()
    all_probs, all_labels_ovf = [], []
    for client_x, broker_x, inter_x, labels in tiny_loader:
        client_x, broker_x, inter_x, labels = (
            client_x.to(DEVICE), broker_x.to(DEVICE),
            inter_x.to(DEVICE),  labels.to(DEVICE)
        )
        logits = _test_model(client_x, broker_x, inter_x)
        loss   = _test_loss_fn(logits, labels)
        _test_optimizer.zero_grad()
        loss.backward()
        _test_optimizer.step()
        all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels_ovf.extend(labels.cpu().numpy())
    if ep % 10 == 0:
        auc = roc_auc_score(all_labels_ovf, all_probs)
        print(f"{ep:>6}  {loss.item():>8.4f}  {auc:>8.4f}")

final_auc = roc_auc_score(all_labels_ovf, all_probs)
if final_auc > 0.85:
    print(f"\n✅ Sanity check PASSED (AUC={final_auc:.3f}) — model can learn, proceed to full training")
else:
    print(f"\n❌ Sanity check FAILED (AUC={final_auc:.3f}) — model has a fundamental issue, debug before training")


Overfit test (50 pos + 50 neg samples):
 Epoch      Loss       AUC
    10    0.0860    0.6952
    20    0.0675    0.8548
    30    0.0894    0.8628
    40    0.0809    0.8572
    50    0.0525    0.9080

✅ Sanity check PASSED (AUC=0.908) — model can learn, proceed to full training


In [11]:
def run_epoch(loader, train=True):
    model.train(train)
    total_loss, all_probs, all_labels = 0.0, [], []

    with torch.set_grad_enabled(train):
        for batch in loader:
            client_x, broker_x, inter_x, labels = batch
            client_x = client_x.to(DEVICE)
            broker_x = broker_x.to(DEVICE)
            inter_x  = inter_x.to(DEVICE)
            labels   = labels.to(DEVICE)

            logits = model(client_x, broker_x, inter_x)
            loss   = loss_fn(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(labels)
            all_probs.extend(torch.sigmoid(logits).cpu().detach().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(all_labels)
    auc      = roc_auc_score(all_labels, all_probs)
    pr_auc   = average_precision_score(all_labels, all_probs)
    return avg_loss, auc, pr_auc


## Gradient Check — Epoch 1
Run one epoch and inspect gradient norms. All should be > 0.0001.
Near-zero norms = vanishing gradients (dead network). Very large norms = exploding gradients.

In [12]:
model.train()
batch = next(iter(train_loader))
client_x, broker_x, inter_x, labels = [t.to(DEVICE) for t in batch]

logits = model(client_x, broker_x, inter_x)
loss   = loss_fn(logits, labels)
optimizer.zero_grad()
loss.backward()

print("Gradient norms after first batch:")
print(f"  {'Layer':<45} {'Grad Norm':>10}  Status")
print(f"  {'-'*60}")
all_ok = True
for name, param in model.named_parameters():
    if param.grad is not None:
        gnorm  = param.grad.norm().item()
        status = "✅" if gnorm > 1e-6 else "❌ DEAD"
        if gnorm <= 1e-6:
            all_ok = False
        print(f"  {name:<45} {gnorm:>10.6f}  {status}")

print()
print(f"Loss on first batch: {loss.item():.4f}")
frac_pos = labels.float().mean().item()
print(f"Positive rate in this batch: {frac_pos*100:.1f}%  (target ~50% with sampler)")
if all_ok:
    print("\n✅ Gradients look healthy — proceed to training")
else:
    print("\n❌ Dead gradients detected — check activation functions and weight init")

# Reset optimizer state before real training
optimizer.zero_grad()


Gradient norms after first batch:
  Layer                                          Grad Norm  Status
  ------------------------------------------------------------
  client_tower.net.0.weight                       0.000428  ✅
  client_tower.net.0.bias                         0.000000  ❌ DEAD
  client_tower.net.1.weight                       0.000039  ✅
  client_tower.net.1.bias                         0.000048  ✅
  client_tower.net.4.weight                       0.000751  ✅
  client_tower.net.4.bias                         0.000000  ❌ DEAD
  client_tower.net.5.weight                       0.000085  ✅
  client_tower.net.5.bias                         0.000132  ✅
  client_tower.net.8.weight                       0.001379  ✅
  client_tower.net.8.bias                         0.000474  ✅
  broker_tower.net.0.weight                       0.000369  ✅
  broker_tower.net.0.bias                         0.000000  ❌ DEAD
  broker_tower.net.1.weight                       0.000044  ✅
  broker_tower.

In [13]:
best_val_auc = 0.0
best_epoch   = 0
history      = []

print(f"\n{'Epoch':>5}  {'Train Loss':>10}  {'Train AUC':>10}  {'Val Loss':>9}  {'Val AUC':>8}  {'Val PR-AUC':>10}")
print("-" * 65)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_auc, train_pr = run_epoch(train_loader, train=True)
    val_loss,   val_auc,   val_pr   = run_epoch(val_loader,   train=False)

    scheduler.step(val_auc)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss, "train_auc": train_auc, "train_pr": train_pr,
        "val_loss":   val_loss,   "val_auc":   val_auc,   "val_pr":   val_pr,
    })

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch   = epoch
        torch.save(model.state_dict(), "two_tower_best.pt")

    print(
        f"{epoch:>5}  {train_loss:>10.4f}  {train_auc:>10.4f}  "
        f"{val_loss:>9.4f}  {val_auc:>8.4f}  {val_pr:>10.4f}"
        + ("  ← best" if epoch == best_epoch else "")
    )

print(f"\nBest model: epoch {best_epoch}  |  val AUC = {best_val_auc:.4f}")



Epoch  Train Loss   Train AUC   Val Loss   Val AUC  Val PR-AUC
-----------------------------------------------------------------
    1      0.0733      0.5217     0.0863    0.5877      0.1445  ← best
    2      0.0704      0.6093     0.0795    0.6192      0.1551  ← best
    3      0.0684      0.6558     0.0786    0.6530      0.1712  ← best
    4      0.0672      0.6761     0.0715    0.6605      0.1735  ← best
    5      0.0658      0.6908     0.0741    0.6750      0.1811  ← best
    6      0.0639      0.7055     0.0746    0.6724      0.1834
    7      0.0624      0.7144     0.0749    0.6836      0.1889  ← best
    8      0.0620      0.7079     0.0699    0.6900      0.1956  ← best
    9      0.0612      0.7190     0.0740    0.6902      0.1909  ← best
   10      0.0609      0.7234     0.0695    0.6903      0.1988  ← best
   11      0.0598      0.7332     0.0713    0.6870      0.2052
   12      0.0586      0.7470     0.0788    0.6842      0.1948
   13      0.0595      0.7363     0.0635  

## Threshold Tuning
Find the optimal decision threshold on the **validation set** instead of using hardcoded 0.5.
With ~11% positives, the real threshold is usually in the 0.2–0.4 range.

In [14]:
model.load_state_dict(torch.load("two_tower_best.pt", map_location=DEVICE))
model.eval()

# Collect val probabilities
val_probs, val_labels_list = [], []
with torch.no_grad():
    for client_x, broker_x, inter_x, labels in val_loader:
        logits = model(client_x.to(DEVICE), broker_x.to(DEVICE), inter_x.to(DEVICE))
        val_probs.extend(torch.sigmoid(logits).cpu().numpy())
        val_labels_list.extend(labels.numpy())

val_probs_arr  = np.array(val_probs)
val_labels_arr = np.array(val_labels_list)

# Sweep thresholds, optimise F1 on positive class
thresholds = np.arange(0.05, 0.90, 0.01)
best_t, best_f1 = 0.5, 0.0
results = []
for t in thresholds:
    preds = (val_probs_arr >= t).astype(int)
    f1    = f1_score(val_labels_arr, preds, zero_division=0)
    results.append((t, f1))
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"Optimal threshold: {best_t:.2f}  (val F1 for 'convert' = {best_f1:.3f})")
print()

# Show threshold vs F1 curve around the optimum
print(f"{'Threshold':>10}  {'F1':>8}")
for t, f1 in results:
    if abs(t - best_t) < 0.08:
        marker = "  ← optimal" if t == best_t else ""
        print(f"{t:>10.2f}  {f1:>8.4f}{marker}")


Optimal threshold: 0.63  (val F1 for 'convert' = 0.280)

 Threshold        F1
      0.55    0.2494
      0.56    0.2522
      0.57    0.2577
      0.58    0.2619
      0.59    0.2645
      0.60    0.2712
      0.61    0.2755
      0.62    0.2787
      0.63    0.2800  ← optimal
      0.64    0.2758
      0.65    0.2646
      0.66    0.2485
      0.67    0.2209
      0.68    0.1878
      0.69    0.0993
      0.70    0.0223


In [15]:
# Final evaluation on test set
test_loss, test_auc, test_pr_auc = run_epoch(test_loader, train=False)

all_probs, all_labels_test = [], []
with torch.no_grad():
    for client_x, broker_x, inter_x, labels in test_loader:
        logits = model(client_x.to(DEVICE), broker_x.to(DEVICE), inter_x.to(DEVICE))
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels_test.extend(labels.numpy())

all_probs_arr  = np.array(all_probs)
all_labels_arr = np.array(all_labels_test)

# Use tuned threshold
all_preds_tuned  = (all_probs_arr >= best_t).astype(int)
all_preds_half   = (all_probs_arr >= 0.50).astype(int)   # for comparison

print("=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"  Loss:              {test_loss:.4f}")
print(f"  AUC-ROC:           {test_auc:.4f}")
print(f"  PR-AUC:            {test_pr_auc:.4f}")
print(f"  Threshold used:    {best_t:.2f}  (tuned on val)")
print()
print(f"--- With tuned threshold ({best_t:.2f}) ---")
print(classification_report(all_labels_arr, all_preds_tuned, target_names=["no convert", "convert"]))
print(f"--- With default threshold (0.50) for comparison ---")
print(classification_report(all_labels_arr, all_preds_half,  target_names=["no convert", "convert"]))

# Probability distribution sanity check
pos_mask = all_labels_arr == 1
neg_mask = all_labels_arr == 0
print("Probability distribution:")
print(f"  Mean prob for positives: {all_probs_arr[pos_mask].mean():.3f}")
print(f"  Mean prob for negatives: {all_probs_arr[neg_mask].mean():.3f}")
print(f"  (These should be clearly different. If similar, model isn't discriminating.)")


TEST SET RESULTS
  Loss:              0.0669
  AUC-ROC:           0.6917
  PR-AUC:            0.2097
  Threshold used:    0.63  (tuned on val)

--- With tuned threshold (0.63) ---
              precision    recall  f1-score   support

  no convert       0.92      0.73      0.81      2713
     convert       0.18      0.49      0.27       341

    accuracy                           0.70      3054
   macro avg       0.55      0.61      0.54      3054
weighted avg       0.84      0.70      0.75      3054

--- With default threshold (0.50) for comparison ---
              precision    recall  f1-score   support

  no convert       0.99      0.29      0.45      2713
     convert       0.15      0.96      0.25       341

    accuracy                           0.37      3054
   macro avg       0.57      0.63      0.35      3054
weighted avg       0.89      0.37      0.43      3054

Probability distribution:
  Mean prob for positives: 0.618
  Mean prob for negatives: 0.524
  (These should be cl